# ⚡ Notebook 1: Circuit Breaker — Fail Fast

When a downstream service is sick, pounding it with more requests makes things worse.
A **circuit breaker** watches the failure rate and, when it crosses a threshold, **opens** — short-circuiting calls and returning an error immediately without touching the downstream.

### Three states
1. `CLOSED` — all good, calls pass through.
2. `OPEN` — too many failures; calls fail instantly.
3. `HALF_OPEN` — after a cool-down, allow one trial call. If it works, close; if it fails, open again.

### Analogy
Your home's electrical breaker trips so the wires don't melt. Same idea.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## A minimal circuit breaker

In [ ]:
import time

class CircuitBreaker:
    def __init__(self, fail_threshold=3, reset_after=2):
        self.state = 'CLOSED'
        self.failures = 0
        self.opened_at = 0
        self.fail_threshold = fail_threshold
        self.reset_after = reset_after

    def call(self, fn, *args, **kw):
        if self.state == 'OPEN':
            if time.time() - self.opened_at >= self.reset_after:
                self.state = 'HALF_OPEN'
                print('→ HALF_OPEN (trial)')
            else:
                raise RuntimeError('circuit OPEN — failing fast')
        try:
            result = fn(*args, **kw)
        except Exception as e:
            self.failures += 1
            if self.state == 'HALF_OPEN' or self.failures >= self.fail_threshold:
                self.state = 'OPEN'
                self.opened_at = time.time()
                print('→ OPEN')
            raise
        # success
        if self.state == 'HALF_OPEN':
            print('→ CLOSED (trial passed)')
        self.state = 'CLOSED'
        self.failures = 0
        return result


## Try it

In [ ]:
import random

def flaky_service(fail=True):
    if fail: raise RuntimeError('downstream down')
    return 'ok'

cb = CircuitBreaker(fail_threshold=3, reset_after=1.5)
for i in range(6):
    try:
        print(i, cb.call(flaky_service, fail=True))
    except Exception as e:
        print(i, 'FAIL —', e)

print('\n... waiting for reset_after ...')
time.sleep(1.6)
# Downstream has recovered
for i in range(3):
    try:
        print(i, cb.call(flaky_service, fail=False))
    except Exception as e:
        print(i, 'FAIL —', e)
